# POC 4: The Handover Extractor

**Pain point:** The week before someone goes on leave or leaves the company, they write a handover document from memory — rushed, incomplete. Critical context disappears: the 'why' behind decisions, the quirks of a client, the informal commitments. The person taking over spends their first two weeks rediscovering what wasn't written down.

**What this notebook shows:** Grounding with the departing person's actual work artifacts (tickets, meeting notes, email threads) extracts a structured handover document they review and approve — not write from scratch. The challenge here is that the grounding material is messy and multi-source.

**You need:** A free Groq API key from https://console.groq.com

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

In [ ]:
!pip install groq -q

In [ ]:
import os
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY') or input('Enter Groq API key: ')

client = Groq(api_key=GROQ_API_KEY)
MODEL = 'qwen/qwen3.8-27b'

def call_llm(system_prompt, user_message, temperature=0.3, max_tokens=1100):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print(f'Model ready: {MODEL}')

In [ ]:
# --- Synthetic grounding material ---
# Represents the messy, multi-source reality of someone's last 90 days.

TICKETS_SUMMARY = """
RECENT TICKETS — Priya Sharma (last 90 days)

[PLAT-1182] API rate limiter refactor — Status: In Review
  Priya refactored the rate limiter to use a sliding window instead of fixed buckets.
  PR is open, waiting for Ravi's sign-off. The old implementation had a race condition
  that only appeared under load (>500 req/s). New impl uses Redis atomic ops.
  NOTE: Ravi is on leave until 12 Nov — PR cannot merge until he returns.

[PLAT-1204] Acme Corp data export — Status: Blocked
  Acme requested a custom CSV export format. Priya built it but it's blocked on
  Acme's legal approving the data processing addendum. Last contact: 3 Oct email
  to James Whitfield (Acme CTO). No response yet. Do not ship without legal sign-off.

[PLAT-1219] Dashboard latency — Status: Done
  Fixed. Root cause was N+1 query in the project listing endpoint.
  Added a composite index on (org_id, created_at). Deployed to prod 28 Sep.

[INFRA-88] Postgres upgrade — Status: Planned
  Postgres 14 → 16 upgrade scheduled for Q4. Priya drafted the runbook (Confluence: INFRA/postgres-16-upgrade).
  Dry run on staging completed — no issues. Prod upgrade window: 15 Nov, 02:00 UTC.
  Requires DBA sign-off from Chen Wei before proceeding.
"""

MEETING_NOTES_SNIPPETS = """
MEETING NOTES (relevant excerpts)

Engineering Sync — 10 Oct 2025:
  Priya flagged that the Acme export feature is a high-priority ask from the sales team —
  Acme is a £180k ARR account. Sales director (Marcus) has been told 'end of Oct' delivery.
  Legal DPA is the only blocker. Priya suggested chasing James Whitfield directly.

Architecture Review — 2 Oct 2025:
  Team agreed to move rate limiter config to environment variables (not hardcoded) in a
  follow-up ticket. Priya said she'd raise it — ticket not yet created.
  This is an unresolved action item.

1:1 with manager — 15 Oct 2025:
  Priya mentioned she's the only person who knows the Redis cluster config.
  Suggested writing it up before she leaves. Not done yet.
"""

EMAIL_SNIPPETS = """
EMAIL THREADS (relevant)

Thread: Acme CSV export — Priya to James Whitfield, 3 Oct:
  'Hi James, the export feature is built and ready to test. We need the signed DPA
   before we can enable it in your production environment. Can you chase your legal team?
   Happy to jump on a call this week if it helps move it forward.'
  No reply received.

Thread: Postgres upgrade — Priya to Chen Wei, 20 Sep:
  'Chen, the staging dry run is done. I've attached the runbook. Can you review
   and sign off when you get a chance? Aiming for 15 Nov prod window.'
  Chen Wei replied 24 Sep: 'Looks good, will formally sign off by end of Oct.'
"""

ARTIFACTS = f"""
=== TICKETS (last 90 days) ===
{TICKETS_SUMMARY}

=== MEETING NOTES (excerpts) ===
{MEETING_NOTES_SNIPPETS}

=== EMAIL THREADS ===
{EMAIL_SNIPPETS}
"""

print('Grounding data loaded.')
print(f'Total artifact size: ~{len(ARTIFACTS.split())} words')

In [ ]:
# --- UNGROUNDED call ---

ungrounded_system = "You are a helpful engineering manager assistant."

ungrounded_query = """
Priya Sharma is going on maternity leave next week. She's a backend engineer.
Generate a handover document for the person taking over her work.
"""

print('=== UNGROUNDED OUTPUT ===')
print(call_llm(ungrounded_system, ungrounded_query))

In [ ]:
# --- GROUNDED call ---

grounded_system = """
You are an engineering knowledge extractor. Given messy, multi-source work artifacts
(tickets, meeting notes, emails), produce a structured handover document.

Sections to produce:
1. Current Projects & Status — for each: what it is, current state, what needs to happen next
2. Open Commitments — promises made to external parties (clients, other teams) with deadlines
3. Blockers & Dependencies — what is waiting on someone else, and who
4. Unresolved Action Items — things that were agreed but not yet done
5. Things Not in the Tickets — informal knowledge, known risks, 'call X for Y' pointers

If something is implied but not confirmed, flag it with [VERIFY].
Do not invent information. If context is missing, say so explicitly.
"""

grounded_query = f"""
Priya Sharma (backend engineer) is going on maternity leave in one week.
Extract a structured handover document from her work artifacts below.
The person taking over has full engineering access but no prior context on her work.

{ARTIFACTS}
"""

print('=== GROUNDED OUTPUT: Handover Document ===')
print(call_llm(grounded_system, grounded_query))

## What just happened

The **ungrounded** output is a generic handover template — current projects, contacts, access credentials, escalation path. Structurally correct. Completely empty.

The **grounded** output names specific items:
- PLAT-1182 is blocked on Ravi who is on leave until 12 Nov
- Acme's export feature is a £180k ARR account and sales promised 'end of Oct' — legal DPA is the only blocker
- The Redis cluster config documentation is missing [VERIFY]
- There's an unraised ticket for rate limiter config from the architecture review

None of those items would appear in a from-memory handover document written in a rushed final week.

**The [VERIFY] flag matters:** the grounded system prompt instructs the model to flag things that are implied but not confirmed. This prevents the handover from becoming a source of false confidence — worse than an incomplete handover is one that's confidently wrong.